# 📊 Análise Exploratória de Dados (EDA) - Predicting Airline Satisfaction
**Competição:** [Kaggle Playground Series Season 6 Episode 10](https://www.kaggle.com/competitions/playground-series-s6e10)  
**Métrica de Avaliação:** ROC AUC Score  
**Objetivo:** Compreender a distribuição dos dados, dados faltantes, cardinalidade e relações entre variáveis e a satisfação do passageiro.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import CATEGORICAL_COLS, SURVEY_RATING_COLS, TARGET_COL
from src.data import load_test, load_train, summarize_dataset

# Estilo de visualização
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 50)

## 1. Carregamento dos Dados

In [ ]:
train_df = load_train()
test_df = load_test()

print(f"Treino: {train_df.shape[0]:,} linhas, {train_df.shape[1]} colunas")
print(f"Teste:  {test_df.shape[0]:,} linhas, {test_df.shape[1]} colunas")
train_df.head()

## 2. Resumo de Tipos e Valores Nulos

In [ ]:
print("=== Resumo de Treino ===")
display(summarize_dataset(train_df, "Treino"))

print("\n=== Resumo de Teste ===")
display(summarize_dataset(test_df, "Teste"))

## 3. Distribuição da Variável Alvo (`satisfaction`)

In [ ]:
target_counts = train_df[TARGET_COL].value_counts(dropna=False, normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, ax=ax, palette="viridis")
ax.set_title("Distribuição Percentual da Variável Alvo (satisfaction)")
ax.set_ylabel("Porcentagem (%)")
for p in ax.patches:
    ax.annotate(
        f"{p.get_height():.2f}%",
        (p.get_x() + p.get_width() / 2.0, p.get_height() / 2),
        ha="center",
        va="center",
        color="white",
        fontweight="bold",
    )
plt.show()

## 4. Variáveis Categóricas vs Satisfação

In [ ]:
cats = [c for c in CATEGORICAL_COLS if c in train_df.columns]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cats[:4]):
    sns.countplot(data=train_df, x=col, hue=TARGET_COL, ax=axes[i], palette="Set2")
    axes[i].set_title(f"Distribuição de {col} por {TARGET_COL}")
    axes[i].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 5. Avaliações de Serviços (Pesquisa de 0 a 5)
Em conjuntos de dados de satisfação de companhias aéreas, as notas dos serviços (wifi, conforto de assento, embarque online, etc.) costumam ter forte poder preditivo.

In [ ]:
survey_cols = [c for c in SURVEY_RATING_COLS if c in train_df.columns]
if survey_cols:
    plt.figure(figsize=(14, 8))
    mean_ratings = train_df.groupby(TARGET_COL)[survey_cols].mean().T
    mean_ratings.plot(kind="barh", figsize=(12, 8), colormap="coolwarm")
    plt.title("Média das Notas de Serviço por Classe de Satisfação")
    plt.xlabel("Nota Média (0 - 5)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

## 6. Atrasos e Distância de Voo

In [ ]:
delay_cols = [
    c
    for c in [
        "Departure Delay in Minutes",
        "Arrival Delay in Minutes",
        "Flight Distance",
    ]
    if c in train_df.columns
]
train_df[delay_cols].describe()

## 7. Próximos Passos
- Rodar modelo Baseline no notebook `02_baseline_model.ipynb`.
- Avaliar estratégias de Feature Engineering (soma de atrasos, média de avaliações, proporções).
- Testar LightGBM, CatBoost e XGBoost com Stratified K-Fold.
- Montar ensemble ou blend de modelos.